# Tutorial 2: From search to silicon: qubit state classification

Tutorial 1 demonstrated the pipeline on a standard benchmark dataset.
This notebook applies the same pipeline to a real physics problem:
binary classification of superconducting qubit readout states from IQ waveforms,
with an explicit on-chip latency constraint.

**Motivation.** Manual hardware-aware model design for qubit readout requires months
of design-space exploration per target platform. Active qubit reset requires a
classification result in under one microsecond, corresponding to roughly 250 clock
cycles at 250 MHz. SNAC-Pack places this budget directly inside the search objective,
so the Pareto front only contains architectures that are feasible by construction.

Slide 4 illustrates why the manual workflow does not scale.
This notebook implements the automated alternative and concludes with hls4ml
converting the best model to an FPGA hardware description project.

In [ ]:
import sys, platform
print(sys.executable)
print(platform.python_version())
import optuna, sqlite3
print('optuna OK')

In [ ]:
import os
import sys
from pathlib import Path

ROOT = Path.cwd().parent.parent
sys.path.insert(0, str(ROOT))

import yaml
import numpy as np
import tensorflow as tf
import pandas as pd
import matplotlib.pyplot as plt

from utils.tf_global_search import GlobalSearchTF
from utils.tf_local_search_combined import combined_local_search_entrypoint
from utils.tf_data_preprocessing import load_and_preprocess_qubit
from utils.tf_visualization import (
    plot_pareto_fronts,
    plot_interactive_2d_pareto,
    plot_3d_pareto_front_heatmap,
)

%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')
tf.get_logger().setLevel('ERROR')
print('TensorFlow:', tf.__version__)

cfg     = yaml.safe_load(open(Path.cwd() / 't2_config.yaml'))
ds_cfg  = cfg['dataset']
s_cfg   = cfg['search']
ss_cfg  = cfg['search_space']
ls_cfg  = cfg['local_search']
out_cfg = cfg['output']

data_dir = str((Path.cwd() / ds_cfg['data_dir']).resolve())

RESULTS_DIR = out_cfg['results_dir']
os.makedirs(RESULTS_DIR, exist_ok=True)
print(f'Results dir : {RESULTS_DIR}')
print(f'Data dir    : {data_dir}')
print(f'n_trials={s_cfg["n_trials"]}, epochs={s_cfg["epochs"]}, hw_metrics={s_cfg["use_hardware_metrics"]}')

## Dataset: superconducting qubit IQ readout

Qubit state is measured by sending a microwave pulse and recording the reflected
signal. The I (in-phase) and Q (quadrature) components are sampled over a time
window. The two classes correspond to the qubit states |0> and |1>.

Each sample consists of `window_size` I samples concatenated with `window_size` Q
samples, producing an 800-dimensional input vector.
The classification network must produce a result within the latency budget.

<details>
<summary>Background: the classification task</summary>

This is a standard binary classification problem: given 800 real numbers representing
a raw readout waveform, predict whether the qubit is in state |0> or |1>.
The input modality is a 1D time series rather than an image, but the network
structure and training procedure are identical to Tutorial 1.

</details>

In [ ]:
x_viz, y_viz, _, _ = load_and_preprocess_qubit(
    data_dir=data_dir,
    start_location=ds_cfg['start_location'],
    window_size=ds_cfg['window_size'],
    subset_size=min(ds_cfg.get('subset_size', 1000), 200),
    normalize=ds_cfg['normalize'],
    flatten=True,
    one_hot=False,
    num_classes=ds_cfg['num_classes'],
)

plt.figure(figsize=(12, 4))
for cls in range(ds_cfg['num_classes']):
    idx = np.where(y_viz == cls)[0]
    if len(idx) > 0:
        plt.plot(x_viz[idx[0]], alpha=0.8, linewidth=1.5, label=f'State |{cls}>')
plt.title('Representative qubit IQ waveforms')
plt.xlabel('Sample index  (I channel: 0-399,  Q channel: 400-799)')
plt.ylabel('Amplitude (normalized)')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()
print(f'Input dimension : {x_viz.shape[1]}  '
      f'(window_size {ds_cfg["window_size"]} x 2 channels)')

## Configuration comparison

The table below shows the differences between the Tutorial 1 and Tutorial 2
configurations. The YAML structure is identical; only the task-specific entries
change.

| Block | Tutorial 1 (MNIST MLP) | Tutorial 2 (qubit) |
|---|---|---|
| `dataset.name` | `mnist` | `qubit` |
| `search.objective_names` | accuracy, BOPs, avg_resource, clock_cycles | same |
| `search.n_trials / epochs` | 8 / 3 | 10 / 10 |
| `search_space.block_types` | `[MLP]` | `[MLP, None]` |
| `synthesis:` section | not present | present |

The `synthesis:` block specifies which local-search precision to load when building
the hls4ml project, and where to write the output.

In [ ]:
with open(Path.cwd() / 't2_config.yaml') as fh:
    print(fh.read())

## Stage 1: Global search with four objectives

The search simultaneously optimizes:

1. **Accuracy** (maximise)
2. **BOPs**: bit operations, a proxy for weight memory (minimize)
3. **avg_resource**: arithmetic mean of LUT%, DSP%, BRAM%, FF% estimates (minimize)
4. **clock_cycles**: predicted inference latency (minimize)

The Pareto front contains all architectures that are not dominated on all four axes.
Model selection is performed by choosing the architecture that meets the application's
latency and resource budget.

In [ ]:
obj_names = s_cfg['objective_names']
max_flags = s_cfg['maximize_flags']
n_folds   = s_cfg.get('n_folds', 1)

searcher = GlobalSearchTF(
    search_space_path=ss_cfg,
    results_dir=RESULTS_DIR,
)

study = searcher.run_search(
    model_type=s_cfg['model_type'],
    n_trials=s_cfg['n_trials'],
    epochs=s_cfg['epochs'],
    dataset=ds_cfg['name'],
    subset_size=ds_cfg.get('subset_size'),
    objectives=obj_names,
    maximize_flags=max_flags,
    use_hardware_metrics=s_cfg['use_hardware_metrics'],
    one_hot=ds_cfg['one_hot'],
    n_folds=n_folds,
    data_dir=data_dir,
    start_location=ds_cfg['start_location'],
    window_size=ds_cfg['window_size'],
    num_classes=ds_cfg['num_classes'],
    normalize=ds_cfg['normalize'],
    flatten=ds_cfg['flatten'],
    storage=None,
)
print('Global search complete.')

## Pareto front

The interactive scatter plot displays all trials.
Hover over any point to inspect the architecture parameters and objective values.
The latency-budget slider in the next cell highlights which architectures satisfy
a given clock-cycle constraint.

In [ ]:
results_df = pd.DataFrame(searcher.results)

if not results_df.empty:
    best = results_df.loc[results_df['performance_metric'].idxmax()]
    print(f'Highest-accuracy trial : {int(best["trial"])}  '
          f'Accuracy : {best["performance_metric"]:.4f}  '
          f'BOPs : {best["bops"]:.2e}')

    obj_info = list(zip(obj_names, max_flags))
    plot_pareto_fronts(results_df, obj_info, save_dir=RESULTS_DIR, show=True)
    plot_interactive_2d_pareto(results_df, obj_info, save_dir=RESULTS_DIR, show=True)
    if len(obj_names) >= 4:
        plot_3d_pareto_front_heatmap(results_df, obj_info, save_dir=RESULTS_DIR, show=True)
else:
    print('No results available. Run the global search cell above first.')

## Latency budget selection

Adjust the slider to set a clock-cycle budget.
Architectures within the budget are shown in teal; those exceeding it are shown in
grey. This shows the best-feasible / closest-feasible selection that the
pipeline applies automatically during constrained search.

In [ ]:
def _plot_latency_budget(results_df, budget_cc):
    if 'clock_cycles' not in results_df.columns:
        print('clock_cycles column not present. '
              'Confirm that use_hardware_metrics is true in the config.')
        return
    feasible   = results_df[results_df['clock_cycles'] <= budget_cc]
    infeasible = results_df[results_df['clock_cycles'] > budget_cc]

    fig, ax = plt.subplots(figsize=(9, 5))
    ax.scatter(infeasible['clock_cycles'], infeasible['performance_metric'],
               alpha=0.4, color='gray', label='Exceeds budget', s=40)
    ax.scatter(feasible['clock_cycles'], feasible['performance_metric'],
               alpha=0.9, color='teal', label='Within budget', s=60, zorder=5)
    ax.axvline(x=budget_cc, color='red', linestyle='--',
               linewidth=2, label=f'Budget: {budget_cc} cc')
    ax.set(xlabel='Clock cycles (predicted latency)',
           ylabel='Accuracy',
           title='Accuracy vs. latency')
    ax.legend()
    ax.grid(True, linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.show()
    print(f'{len(feasible)} of {len(results_df)} architectures satisfy the '
          f'{budget_cc}-cycle budget.')

if not results_df.empty and 'clock_cycles' in results_df.columns:
    cc_min     = int(results_df['clock_cycles'].min())
    cc_max     = int(results_df['clock_cycles'].max())
    cc_step    = max(1, (cc_max - cc_min) // 20)
    cc_default = int(cc_min + (cc_max - cc_min) * 0.5)

    try:
        import ipywidgets as widgets
        widgets.interact(
            lambda budget_cc: _plot_latency_budget(results_df, budget_cc),
            budget_cc=widgets.IntSlider(
                min=cc_min, max=cc_max, step=cc_step, value=cc_default,
                description='Budget (cc):',
                style={'description_width': 'initial'},
                layout=widgets.Layout(width='500px'),
            ),
        )
    except ImportError:
        _plot_latency_budget(results_df, cc_default)
else:
    print('Run the global search cell first.')

## Stage 2: Combined QAT and pruning

Block-based models use k-fold cross-validation during global search.
For these models, SNAC-Pack applies the combined local search strategy, in which
quantization-aware training and magnitude pruning are interleaved within each
training iteration rather than run as separate sequential phases.
The implementation is in `utils/tf_local_search_combined.py`.

In [ ]:
LOCAL_RESULTS_DIR = os.path.join(RESULTS_DIR, 'local_search_combined')
LOCAL_CONFIG_PATH = os.path.join(RESULTS_DIR, 'local_search_config.yaml')
ARCH_YAML_PATH    = os.path.join(RESULTS_DIR, 'best_model_for_local_search.yaml')

local_search_settings = {
    'pruning_settings': {
        'iterations':           ls_cfg['pruning_iterations'],
        'epochs_per_iteration': ls_cfg['pruning_epochs'],
        'pruning_rate':         ls_cfg['pruning_rate'],
    },
    'qat_settings': {
        'epochs':          ls_cfg['qat_epochs'],
        'precision_pairs': ls_cfg['precision_pairs'],
    },
}
with open(LOCAL_CONFIG_PATH, 'w') as fh:
    yaml.dump(local_search_settings, fh)

x_train, y_train, x_test, y_test = load_and_preprocess_qubit(
    data_dir=data_dir,
    start_location=ds_cfg['start_location'],
    window_size=ds_cfg['window_size'],
    subset_size=ds_cfg.get('subset_size'),
    normalize=ds_cfg['normalize'],
    flatten=ds_cfg['flatten'],
    one_hot=True,
    num_classes=ds_cfg['num_classes'],
)

if not os.path.exists(ARCH_YAML_PATH):
    print(f'Architecture YAML not found: {ARCH_YAML_PATH}')
    print('Complete the global search first.')
    combined_df = pd.DataFrame()
else:
    combined_df = combined_local_search_entrypoint(
        architecture_yaml_path=ARCH_YAML_PATH,
        local_search_config_path=LOCAL_CONFIG_PATH,
        dataset=(x_train, y_train, x_test, y_test),
        results_dir=LOCAL_RESULTS_DIR,
    )

In [ ]:
if isinstance(combined_df, pd.DataFrame) and not combined_df.empty:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for prec in combined_df['Precision'].unique():
        sub = combined_df[combined_df['Precision'] == prec]
        axes[0].plot(sub['EffectiveBOPs'], sub['Accuracy'],
                     marker='o', linewidth=2, label=prec)
        axes[1].plot(sub['Sparsity'], sub['Accuracy'],
                     marker='o', linewidth=2, label=prec)
    axes[0].set(xlabel='Effective BOPs', ylabel='Accuracy',
                title='Accuracy vs. effective BOPs')
    axes[0].set_xscale('log')
    axes[0].invert_xaxis()
    axes[0].legend(title='Precision')
    axes[0].grid(True, linestyle='--', alpha=0.6)
    axes[1].set(xlabel='Sparsity', ylabel='Accuracy',
                title='Accuracy vs. sparsity')
    axes[1].legend(title='Precision')
    axes[1].grid(True, linestyle='--', alpha=0.6)
    plt.tight_layout()
    plt.show()
else:
    print('No local search results available.')

## Stage 3: hls4ml synthesis

This stage loads the best QAT model at the precision specified in `t2_config.yaml`
and generates an HLS C++ project using hls4ml.
The project can be compiled for C-simulation and then synthesised with Vivado to
produce a bitstream and a post-synthesis resource and latency report.

**Note:** the `hls_model.build()` call requires Vivado in the system PATH.
If Vivado is not available, the cells below will still generate the HLS project;
the build step will print an informative message and exit cleanly.

In [ ]:
try:
    from utils.tf_synthesis import run_synthesis_from_config

    syn_cfg = cfg.get('synthesis', {})
    if not syn_cfg:
        print('No synthesis section found in t2_config.yaml.')
    else:
        model, input_shape, hls_model = run_synthesis_from_config(
            config=cfg,
            base_dir=Path.cwd(),
        )
        print(f'Loaded model: {syn_cfg["total_bits"]}b{syn_cfg["int_bits"]}i, '
              f'input shape {input_shape}')
        print(f'HLS project written to: {syn_cfg["hls_output_dir"]}')
except Exception as exc:
    print(f'Synthesis setup error: {exc}')
    hls_model = None

In [ ]:
# C-simulation compile
if hls_model is not None:
    try:
        hls_model.compile()
        print('HLS model compiled successfully.')
    except Exception as exc:
        print(f'HLS compile error: {exc}')
else:
    print('No hls_model object. Run the synthesis cell above first.')

In [ ]:
# Vivado RTL synthesis (requires Vivado in PATH)
if hls_model is not None:
    try:
        hls_model.build()
        print('Vivado synthesis complete. '
              'Resource and latency report available in the HLS project directory.')
    except Exception as exc:
        print(f'Vivado synthesis unavailable: {exc}')
        print('The HLS project is still valid and can be synthesised on a '
              'machine with Vivado installed.')
else:
    print('No hls_model object. Run the synthesis cell above first.')

---
## (Optional) Planner demonstration: English specification to YAML config

*This section is a presenter demonstration and does not need to be run by
participants.*

The SNAC-Pack pipeline is also exposed as Model Context Protocol (MCP) tools,
allowing an AI agent to drive the entire search from a natural-language specification
(slide 19).

The cell below calls the search planner directly to show how a structured
specification is converted into a complete YAML configuration file.
This is the same operation that the MCP `create_search_config` tool performs
when invoked by an agent.

In [ ]:
try:
    from utils.search_planner import build_search_config, write_search_config

    dataset_spec = {
        'dataset_name': 'qubit',
        'dataset_type': 'signal',
        'input_dim':    800,
        'num_classes':  2,
        'task':         'binary_classification',
    }
    constraints = {
        'latency_budget_cc': 100,
        'accuracy_target':   0.95,
        'target_board':      'zcu102',
        'n_trials':          50,
    }

    config = build_search_config(dataset_spec=dataset_spec, constraints=constraints)
    demo_dir = os.path.join(RESULTS_DIR, 'planner_demo')
    os.makedirs(demo_dir, exist_ok=True)
    config_path = write_search_config(
        config,
        output_path=os.path.join(demo_dir, 'generated_config.yaml'),
    )
    print(f'Generated configuration saved to: {config_path}')
    print()
    with open(config_path) as fh:
        print(fh.read())
except Exception as exc:
    print(f'Planner demonstration failed: {exc}')

## Summary

| Stage | Description |
|---|---|
| Dataset | Superconducting qubit IQ readout, 800-dimensional input, binary classification |
| Global search | NSGA-II with `rule4ml` surrogate over accuracy, BOPs, resources, and latency |
| Budget selection | Interactive Pareto visualization with adjustable latency constraint |
| Local search | Combined QAT and iterative magnitude pruning |
| Synthesis | hls4ml HLS project generation and optional Vivado RTL synthesis |
| Planner | Search planner converts a structured specification to a complete YAML config |

The complete pipeline, from dataset to synthesisable FPGA model, is driven
by a single YAML configuration file and can also be invoked from an AI agent
through the MCP tool interface.